In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [2]:
df_main = pd.read_csv("~/mission_sih/mission_sih/data/nasa_data_cleaned.csv")


In [3]:

df_cat = pd.read_csv("~/mission_sih/mission_sih/data/land_use_data.csv")

In [4]:
df_main['acq_date'] = pd.to_datetime(df_main['acq_date'])
df_main['year'] = df_main['acq_date'].dt.year

In [5]:

EARTH_RADIUS_M = 6371000
RADIUS_LIMIT_M = 2000

In [6]:
df_cat.info()

<class 'pandas.DataFrame'>
RangeIndex: 17650701 entries, 0 to 17650700
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   year       int64  
 1   land_type  str    
 2   latitude   float64
 3   longitude  float64
dtypes: float64(2), int64(1), str(1)
memory usage: 538.7 MB


In [7]:
df_main['category'] = pd.Series([np.nan] * len(df_main), dtype='object')
df_main['match_dist_m'] = np.nan

In [8]:
for yr, group in df_main.groupby('year'):
    cat_group = df_cat[df_cat['year'] == yr]
    if cat_group.empty:
        continue  # no category data for this year, leave unmatched

    cat_rad = np.radians(cat_group[['latitude', 'longitude']].values)
    main_rad = np.radians(group[['latitude', 'longitude']].values)

    nn = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
    nn.fit(cat_rad)
    dist, idx = nn.kneighbors(main_rad)

    dist_m = dist.flatten() * EARTH_RADIUS_M
    matched_cat = cat_group.iloc[idx.flatten()]['land_type'].values.astype(object)

    # apply 1km cutoff
    matched_cat = np.where(dist_m <= RADIUS_LIMIT_M, matched_cat, np.nan)

    df_main.loc[group.index, 'category'] = matched_cat
    df_main.loc[group.index, 'match_dist_m'] = dist_m

In [9]:
df_cat2 = pd.read_csv("~/mission_sih/mission_sih/data/osm_data_cleaned.csv")

In [10]:
# df_cat2 has lat, lon, category — no year column, direct global search
cat2_rad = np.radians(df_cat2[['latitude', 'longitude']].values)
main_rad = np.radians(df_main[['latitude', 'longitude']].values)

nn2 = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
nn2.fit(cat2_rad)
dist2, idx2 = nn2.kneighbors(main_rad)

dist2_m = dist2.flatten() * EARTH_RADIUS_M
matched_cat2 = df_cat2.iloc[idx2.flatten()]['osm_category'].values.astype(object)

RADIUS_LIMIT_M_2 = 2000

# candidate is valid only if within 2km
valid2 = dist2_m <= RADIUS_LIMIT_M_2

# overwrite only if: valid AND (no previous match OR this one is closer)
prev_dist = df_main['match_dist_m'].values
no_prev_match = df_main['category'].isna().values
closer = dist2_m < prev_dist

should_replace = valid2 & (no_prev_match | closer)

df_main.loc[should_replace, 'category'] = matched_cat2[should_replace]
df_main.loc[should_replace, 'match_dist_m'] = dist2_m[should_replace]

In [11]:
df_main.head(10)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,confidence,bright_t31,frp,daynight,type,year,category,match_dist_m
0,23.04102,92.52061,336.69,0.62,0.71,2018-04-01,606,1,296.72,6.39,0,0,2018,Forest,307.030507
1,22.58705,92.50738,329.40,0.63,0.72,2018-04-01,606,1,299.01,7.35,0,0,2018,NaN,3249.258609
2,22.58846,92.51312,337.59,0.63,0.72,2018-04-01,606,1,299.01,6.89,0,0,2018,NaN,2885.821332
3,22.50259,92.55136,330.26,0.63,0.72,2018-04-01,606,1,299.59,8.80,0,0,2018,Forest,313.398197
4,22.50408,92.55746,346.15,0.63,0.72,2018-04-01,606,1,298.87,10.10,0,0,2018,Forest,548.842875
5,22.41149,92.65405,330.42,0.63,0.72,2018-04-01,606,0,300.69,3.30,0,0,2018,Forest,102.962301
6,22.12835,92.67033,330.93,0.63,0.72,2018-04-01,606,1,297.33,30.87,0,0,2018,Forest,421.911597
7,22.12196,92.67233,355.08,0.63,0.72,2018-04-01,606,1,300.37,30.87,0,0,2018,Agriculture,197.365346
8,22.10355,92.81759,338.69,0.62,0.72,2018-04-01,606,1,300.20,7.17,0,0,2018,Forest,523.123014
9,25.07154,93.79645,326.75,0.47,0.64,2018-04-01,606,1,281.35,8.20,0,0,2018,Forest,288.904439


In [12]:
df_main["category"].isnull().sum()

np.int64(988074)

In [13]:
df_main.to_csv("~/mission_sih/mission_sih/data/final_dataset.csv", index=False)